# How does Will.y work?

We have built this notebook to allow an easier reading and understanding of the Solidity code of the smart contract (i.e. WillyVault), that you can find in *'willyDEMO.sol'*.  
This is an extra step, all major info can be found in the original pdf report.  

# So, what is WillyVault?

WillyVault holds ETH and allocates it to beneficiaries after shared approval and a veto window.

# Let's look at the code, step by step:

The code provides the main features, here summarized: 
- **Set up the vault**: create the vault that stores the ETH, using constructor.
- **Define the four phases**: literally define the state of the Will: Active, Opening, Window and Finalized.
- **Store the plan and deposit ETH**: update the heirs' plan and deposit money.
- **Keep the owner in control**: allows special powers to the owner (who is authenticated).
- **Collect matching approvals**: the main feature that allows the non-owner to initiate the Will process.
- **Allow time for a veto**: additional customizable feature to gurantee appropriate veto powers to the owner.
- **Finalize the amounts**: calculates the shares in wei.
- **Let each beneficiary claim**: final step, send the funds to the heirs.
## 1. Set up the vault

The `constructor` sets the roles, durations and initial plan:

- **Owner:** manages the plan, withdraws and can cancel before finalization.
- **Three designees:** two must approve a succession.
- **Notary and substitute:** either provides the one required notary approval.
- **Beneficiaries:** receive the percentages chosen in the plan.

Designees must be distinct and separate from the two distinct notaries. Roles cannot be changed after deployment. `vetoWindow` and `pauseDuration` are also set at deployment; a two-minute window is a demo choice, not a fixed rule.

## 2. The four phases

```text
Active → Opening → Window → Finalized
```

| Phase | Meaning |
|---|---|
| `Active` | The owner can manage funds and the plan. |
| `Opening` | Proposals collect approvals; owner actions remain available. |
| `Window` | A proposal has enough approvals; the veto period runs. |
| `Finalized` | Amounts are fixed; beneficiaries can claim. |

`cancel()` returns any unfinished process to Active. Passing time alone never changes the phase: someone must submit a transaction.

## 3. Store the plan and deposit ETH

`_setPlan()` stores two matching arrays: beneficiary addresses and shares in basis points. `6000` means 60%; `4000` means 40%. Shares must be positive and total `10_000` (100%). The plan allows 1–16 entries and rejects zero addresses.

`planVersion` increases whenever a plan is stored. `getPlan()` returns the current arrays.

Anyone can add ETH through `deposit()` or a plain transfer handled by `receive()`. Both call `_deposit()`. Ordinary deposits are rejected after Finalized. The ETH belongs to the vault's balance: `address(this).balance`.

## 4. Keep the owner in control

`onlyOwner` checks that `msg.sender`, the caller, is the owner.

| Function | What it does |
|---|---|
| `updatePlan()` | Replaces the plan in Active/Opening and invalidates approvals. |
| `withdraw()` | Withdraws in Active/Opening without changing shares or approvals. |
| `cancel()` | Cancels before finalization and invalidates approvals. |
| `cancelAndWithdraw()` | Cancels and withdraws in one transaction before finalization. |

`_withdraw()` checks the recipient and amount before sending ETH. If the combined cancellation and withdrawal fails, the whole transaction rolls back. During Window, the owner must cancel before changing the plan or use `cancelAndWithdraw()` to withdraw.

## 5. Collect matching approvals

Each authorized caller submits:

```solidity
openSuccession(caseId_, planVersion_, attestationHash_, approvalCycle_, expiresAt_)
```

All approvers must use the same five values: case identifier, current plan version, document hash, current approval cycle and shared expiry. The expiry must be in the future, at most seven days ahead when submitted.  

`getProposalId()` hashes these values together with the chain ID and vault address. `proposals` stores the counts; `hasApproved` prevents duplicate approvals. Independent proposals can collect approvals without blocking each other.

A proposal needs two distinct designees AND either notary. Both notary addresses together still count as one notary approval. Reaching this threshold selects the proposal and starts Window.

Cancellation increases `approvalCycle`; a plan update increases `planVersion` and `approvalCycle` . Old approvals remain stored but cannot authorize the new cycle or plan. The document hash identifies a document; it does not prove death.

## 6. Allow time for a veto

The selected proposal starts a deadline:

```solidity
windowEnd = block.timestamp + vetoWindow;
```

Any designee can call `pause()` once per succession, before the deadline. It adds `pauseDuration` to `windowEnd`. Only one extension is allowed per succession, regardless of which designee requests it.  

The owner can cancel until `finalize()` actually executes, even after the deadline. Once the deadline is reached, cancellation and finalization may both be possible: transaction order matters. The selected process now follows this veto deadline, not the original proposal expiry.

## 7. Finalize the amounts

`finalize()` can be called by anyone in Window once the deadline is reached. The caller cannot change the recipients or shares.

The function reads the current balance and calculates each allocation:

```solidity
uint256 amount = (total * _bps[i]) / TOTAL_BPS;
claimable[_beneficiaries[i]] += amount;
```

ETH is counted in whole wei. Division rounds down, so the last plan entry receives the remaining amount. The function stores `finalizedTotal` and sets the phase to Finalized; it does not send payments yet.

Example: deposit 10 ETH and withdraw 4 ETH before succession. With a 60/40 plan, the remaining 6 ETH gives 3.6 ETH and 2.4 ETH. Shares apply to the actual final balance.

## 8. Let each beneficiary claim

`claim()` reads `claimable[msg.sender]`. Only the beneficiary calling it receives that address's allocation.

```solidity
        // Effects
        claimable[msg.sender] = 0;
        emit Claimed(msg.sender, amount);

        // Interaction
        (bool ok,) = msg.sender.call{value: amount}("");
        if (!ok) revert TransferFailed();
```

The amount is set to zero before sending ETH, preventing a second payment of the same allocation. If the transfer fails, all changes roll back. Each beneficiary claims independently.

The shared `nonReentrant` modifier blocks nested calls to protected functions. Events record actions for an interface; custom errors reject invalid calls. These checks are part of the demo implementation, not evidence of an audit.